In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

df = pd.read_csv("week10_students.csv")
df.head()

,student_id,name,gender,program,district,attendance_pct,quiz_avg,assignments_avg,score
0,AUCA001,Aline Uwase,F,Information Systems,Gasabo,55,31.0,35.0,28.6
1,AUCA002,Eric Iradukunda,M,Software Engineering,Musanze,73,61.1,76.7,64.5
2,AUCA003,Diane Uwizeyimana,F,Networking,Muhanga,92,35.2,65.5,52.7
3,AUCA004,Jean Bosco Ikirezi,M,Accounting,Kicukiro,69,65.2,54.2,68.3
4,AUCA005,Patrick Ingabire,M,Information Systems,Huye,88,39.3,42.9,42.3


In [2]:
print("Shape:", df.shape)
print()
print(df[["score", "quiz_avg", "assignments_avg", "attendance_pct"]].corr()["score"])

# Comment: quiz_avg looks most useful because it has the strongest
# correlation with score. attendance_pct is the weakest.

Shape: (150, 9)

score              1.000000
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64


In [3]:
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Train size:", len(X_train))   # 112
print("Test size :", len(X_test))    # 38

# Comment: we hide 25% of students (the test set) so we can check the model
# on students it has never seen. Testing on the same students it trained on
# would look better than reality, because the model could just memorise them.

Train size: 112
Test size : 38


In [4]:
reg = LinearRegression()
reg.fit(X_train, y_train)

for name, coef in zip(features, reg.coef_):
    print(f"{name:18s} coefficient = {coef:.3f}")
print(f"{'intercept':18s} = {reg.intercept_:.3f}")

# Comment: the quiz_avg coefficient is the change in predicted score for one
# extra quiz point, with assignments and attendance held fixed.
# (Read the value printed above, e.g. 0.55 means about 0.55 exam points.)

quiz_avg           coefficient = 0.579
assignments_avg    coefficient = 0.365
attendance_pct     coefficient = 0.113
intercept          = -4.428


In [5]:
y_pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE = {mae:.2f} points")   # checkpoint about 3.3
print(f"R2  = {r2:.2f}")           # checkpoint about 0.89

# Comment (plain language for a dean):
# MAE: on average the prediction is off by about 3 points out of 100
#      for students the model has never seen.
# R2: quiz, assignments and attendance explain about 89% of the differences
#     between students' scores. The other ~11% is due to things the model cannot see.

MAE = 3.33 points
R2  = 0.89


In [6]:
new_student = pd.DataFrame({
    "quiz_avg": [70],
    "assignments_avg": [75],
    "attendance_pct": [90],
})
print("Predicted score (new student):", round(reg.predict(new_student)[0], 1))

# Your own student, with invented numbers (change them)
me = pd.DataFrame({
    "quiz_avg": [65],
    "assignments_avg": [80],
    "attendance_pct": [95],
})
print("Predicted score (me):", round(reg.predict(me)[0], 1))

Predicted score (new student): 73.7
Predicted score (me): 73.2


In [7]:
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())   # 1 = pass, 0 = fail

yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, yc, test_size=0.25, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64


LogisticRegression(max_iter=1000)

In [8]:
yc_pred = clf.predict(Xc_test)

acc = accuracy_score(yc_test, yc_pred)
baseline = yc_test.mean()   # lazy model: predict "pass" for everyone

print(f"Model accuracy        : {acc:.3f}")
print(f"Lazy baseline accuracy: {baseline:.3f}")
print("Confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(yc_test, yc_pred))
# Layout: [[TN FP]
#          [FN TP]]   (0 = fail, 1 = pass)

# Comment: the two accuracies are nearly identical because most students pass,
# so "predict pass for everyone" already scores high. But our model is NOT useless.
# The lazy model catches 0 failing students (it never predicts "fail").
# In the confusion matrix, the top-left cell (TN) shows how many failing
# students our model correctly flags. Those are the at-risk students we want to help.
# Lesson: with imbalanced classes, read the confusion matrix, not accuracy alone.

Model accuracy        : 0.895
Lazy baseline accuracy: 0.895
Confusion matrix (rows = actual, cols = predicted):
[[ 3  1]
 [ 3 31]]


In [9]:
def test_r2(X_in):
    Xtr, Xte, ytr, yte = train_test_split(X_in, y, test_size=0.25, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

X_v1 = df[["quiz_avg"]]
X_v2 = df[features]
X_v3 = pd.get_dummies(df[features + ["program"]], columns=["program"], drop_first=True)

results = pd.DataFrame({
    "version": ["1. quiz_avg only", "2. three numeric features", "3. three + program (one-hot)"],
    "test_R2": [test_r2(X_v1), test_r2(X_v2), test_r2(X_v3)],
})
print(results.round(3).to_string(index=False))   # checkpoint about 0.62 -> 0.89 -> 0.88

# Comment: going from 1 to 3 numeric features helped a lot (about 0.62 to 0.89),
# because assignments and attendance carry real information about the score.
# Adding program (one-hot) did not help (flat or slightly lower): it adds no new
# signal once we know the three numeric features, and extra columns can add noise
# on a small dataset (150 rows).
# Lesson: more features is not automatically better. Test on unseen data
# to check whether new information really helps.

                     version  test_R2
            1. quiz_avg only    0.624
   2. three numeric features    0.887
3. three + program (one-hot)    0.880


In [ ]:
# Plotly needs a categorical colour, so we use a labelled copy for plotting
df_plot = df.copy()
df_plot["passed"] = df_plot["passed"].map({1: "Pass", 0: "Fail"})

# (a) Histogram of scores, 12 bins
fig1 = px.histogram(df_plot, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

# (b) Scatter: quiz_avg vs score, coloured by pass/fail
fig2 = px.scatter(df_plot, x="quiz_avg", y="score", color="passed",
                  hover_name="name", title="2. Who is at risk")
fig2.show()

# (c) Bar chart: average score per district
district_avg = df_plot.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(district_avg, x="district", y="score", title="3. Where to act")
fig3.show()

In [12]:
fig2.write_html("class_dashboard.html")

# In Colab, download it to your computer:
from google.colab import files
files.download("class_dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
fig4 = px.box(df, x="program", y="score", color="program",
              title="Score distribution by program")
fig4.show()

print(df.groupby("program")["score"].agg(["std", "min", "max"]).round(1))

# Two sentences (fill in using the chart and the table above):
# The most consistent program is ____ because its box is the narrowest (lowest std).
# The program with the widest spread is ____, where scores range from ____ to ____.

                       std   min   max
program                               
Accounting            11.0  43.0  89.5
Information Systems   12.3  28.6  84.4
Networking            10.8  41.9  89.4
Software Engineering  13.8  37.2  92.9


MODEL CARD: Student Score Predictor

1. Purpose
Predicts a student's final score (0 to 100) from their quiz average,
assignments average and attendance. It is meant for lecturers and the
faculty to spot students who may need early support.

2. Data and features
150 AUCA students (week10_students.csv, cleaned). Inputs: quiz_avg,
assignments_avg, attendance_pct. Target: score. Trained on 112 students
and tested on 38 unseen students (test_size=0.25, random_state=42).

3. Performance
On unseen students the model is off by about 3.3 points on average (MAE),
and it explains about 89% of the differences between students' scores
(R2 = 0.89). [Replace with your own printed numbers if they differ.]

4. Risks
A wrong prediction could label a capable student as "at risk" (stigma,
lower expectations) or miss a student who really needs help. The model
only sees three numbers. It does not know about illness, family problems,
language or effort.
Use it to: start a conversation and offer tutoring or support early.
Do NOT use it to: decide grades, pass/fail, scholarships or discipline,
or to replace the lecturer's judgement. It was trained on 150 students
from one course, so it may not work for other groups.